# ESRGAN 4x / 23-block / Simple Degradation 训练（Colab 一体化）

本 Notebook 包含从 **环境初始化 → 退化数据生成 → 模型训练 → 损失曲线** 的完整流程。  
在 Colab 中**从上到下运行所有 cell** 即可完成一次完整实验。

### 存储方案
| 位置 | 内容 |
|------|------|
| UCSD OneDrive | `dataset.zip`（含 `dataset/` + `pretrained_models/`） |
| UCSD Google Drive (5 GB) | `saved_models/`, `results/`（持久化） |
| GitHub `main` | 代码 + `APISR_tools/` |
| Colab 本地 SSD | 下载解压后的 dataset（临时） |

In [ ]:
# ============================================================
# 0. 全局配置
# ============================================================

GITHUB_REPO      = 'https://github.com/liqiqinaOH7/AWM.git'
BRANCH           = 'main'
ONEDRIVE_ZIP_URL = 'https://ucsdcloud-my.sharepoint.com/:u:/g/personal/xil326_ucsd_edu/IQDFdjcJu1AAQ67p-ln0IB85AdUSjend_ov_nfx_A6DNlX0?download=1'
DRIVE_ROOT       = '/content/drive/MyDrive/AWM'
PROJECT_DIR      = '/content/AWM'

In [ ]:
# ============================================================
# 1. 挂载 Google Drive
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

import os
for d in ['saved_models', 'results']:
    os.makedirs(f'{DRIVE_ROOT}/{d}', exist_ok=True)
print(f'Drive 就绪: {DRIVE_ROOT}')

In [ ]:
# ============================================================
# 2. 从 OneDrive 下载 dataset.zip（含 dataset/ + pretrained_models/）
# ============================================================
import os, time

zip_path = '/content/dataset.zip'
marker   = '/content/dataset/highres/original'

if os.path.isdir(marker):
    print('dataset 已存在，跳过下载。')
else:
    print('正在从 OneDrive 下载 dataset.zip（~3.4 GB）...')
    t0 = time.time()
    !wget -q --show-progress -O "{zip_path}" "{ONEDRIVE_ZIP_URL}"
    elapsed = time.time() - t0
    if os.path.isfile(zip_path) and os.path.getsize(zip_path) > 1_000_000:
        print(f'下载完成: {os.path.getsize(zip_path)/1e9:.2f} GB, {elapsed:.0f}s，解压中...')
        !unzip -q -o "{zip_path}" -d /content/
        os.remove(zip_path)
        print('解压完成。')
    else:
        print('⚠ 下载可能失败，请检查 OneDrive 链接权限和 URL。')

In [ ]:
# ============================================================
# 2b. 备用：如果 wget 下载失败，手动上传 dataset.zip 后运行此 cell
# ============================================================
# from google.colab import files
# uploaded = files.upload()
# import os
# if os.path.isfile('/content/dataset.zip'):
#     !unzip -q -o /content/dataset.zip -d /content/
#     os.remove('/content/dataset.zip')
#     print('解压完成。')

In [ ]:
# ============================================================
# 3. 拉取 GitHub 代码 + 创建软链接
# ============================================================
import os

if os.path.isdir(PROJECT_DIR):
    !cd {PROJECT_DIR} && git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {GITHUB_REPO} {PROJECT_DIR}

os.chdir(PROJECT_DIR)
print(f'工作目录: {os.getcwd()}')

links = {
    'dataset':           '/content/dataset',
    'pretrained_models': '/content/pretrained_models',
    'saved_models':      f'{DRIVE_ROOT}/saved_models',
    'results':           f'{DRIVE_ROOT}/results',
}
for name, target in links.items():
    lp = os.path.join(PROJECT_DIR, name)
    if os.path.islink(lp):
        if os.readlink(lp) == target: continue
        os.unlink(lp)
    elif os.path.isdir(lp): continue
    os.symlink(target, lp)
    print(f'  链接: {name} → {target}')

!pip install -q pyiqa tqdm
print('\n环境初始化完成。')

In [ ]:
# ============================================================
# 4. 生成简化版 4x LR 数据集（仅 bicubic 下采样，无滤波）
#    如果已存在则跳过
# ============================================================
import os, glob, cv2
from tqdm import tqdm

hr_dir = 'dataset/highres/original'
lr_dir = 'dataset/lowres_4x_simple/original'
scale  = 4

os.makedirs(lr_dir, exist_ok=True)
hr_image_paths = sorted(glob.glob(os.path.join(hr_dir, '*.[jp][pn]*g')))
existing_lr = len(os.listdir(lr_dir)) if os.path.isdir(lr_dir) else 0

if existing_lr >= len(hr_image_paths) and existing_lr > 0:
    print(f'lowres_4x_simple 已有 {existing_lr} 张，跳过生成。')
else:
    print(f'正在生成简化版 4x LR（{len(hr_image_paths)} 张 HR）...')
    count = 0
    for hr_path in tqdm(hr_image_paths, desc='4x bicubic'):
        img = cv2.imread(hr_path)
        if img is None: continue
        h, w = img.shape[:2]
        img_lr = cv2.resize(img, (w // scale, h // scale), interpolation=cv2.INTER_CUBIC)
        fn = os.path.basename(hr_path)
        if fn.lower().endswith('.png'): fn = fn[:-4] + '.jpg'
        cv2.imwrite(os.path.join(lr_dir, fn), img_lr, [int(cv2.IMWRITE_JPEG_QUALITY), 85])
        count += 1
    print(f'完成: {count} 张 LR 保存到 {lr_dir}')

In [ ]:
# ============================================================
# 5. 验证数据集
# ============================================================
import glob, os
import torch, cv2

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "N/A"}')

hr_count = len(glob.glob('dataset/highres/original/*.[jp][pn]*g'))
lr_count = len(glob.glob('dataset/lowres_4x_simple/original/*.[jp][pn]*g'))
print(f'HR: {hr_count} 张 | LR (4x simple): {lr_count} 张')

pt = glob.glob('pretrained_models/*.pth')
print(f'预训练权重: {[os.path.basename(f) for f in pt]}')

assert hr_count > 0, '未检测到 HR 图片'
assert lr_count > 0, '未检测到 LR 图片'
print('\n✅ 数据就绪！')

---
## 训练部分

In [ ]:
# ============================================================
# 6. Import
# ============================================================
import os, sys, glob, random
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

apisr_tools_path = os.path.abspath('APISR_tools')
if apisr_tools_path not in sys.path:
    sys.path.append(apisr_tools_path)

from architecture.rrdb import RRDBNet
from architecture.discriminator import UNetDiscriminatorSN
from loss.perceptual_loss import PerceptualLoss
from loss.anime_perceptual_loss import Anime_PerceptualLoss
from loss.gan_loss import GANLoss

print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())

In [ ]:
# ============================================================
# 7. 超参数配置
# ============================================================

UPSCALE_MODE = '4x'
SCALE = 4

LR_DIR = 'dataset/lowres_4x_simple/original'
HR_DIR = 'dataset/highres/original'
PRETRAINED_DIR = 'pretrained_models'

BATCH_SIZE    = 8
PATCH_SIZE    = 256       # HR 裁剪尺寸，LR 对应 64
NUM_EPOCHS    = 150
LEARNING_RATE = 1e-5
SAVE_FREQ     = 10

NUM_BLOCK = 23            # APISR 官方 4x 架构

LR_STEP_SIZE = 30
LR_GAMMA     = 0.5
LR_MIN       = 1e-6

CHECKPOINT_NAME = 'ESRGAN_4x_23block_simple'

WARMUP_EPOCHS    = 5
WEIGHT_PIXEL     = 10.0
WEIGHT_VGG       = 0.05
WEIGHT_DANBOORU  = 0.025
WEIGHT_GAN       = 1.0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
# ============================================================
# 8. 数据集
# ============================================================

class AnimeSRDataset(Dataset):
    def __init__(self, lr_dir, hr_dir, patch_size=256, scale=4):
        self.lr_paths = sorted(glob.glob(os.path.join(lr_dir, '*.*')))
        self.hr_paths = sorted(glob.glob(os.path.join(hr_dir, '*.*')))
        self.patch_size = patch_size
        self.scale = scale
        print(f'LR: {len(self.lr_paths)} | HR: {len(self.hr_paths)}')
        assert len(self.lr_paths) == len(self.hr_paths), 'LR / HR 数量不匹配！'

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        hr_img = cv2.imread(self.hr_paths[idx])
        lr_img = cv2.imread(self.lr_paths[idx])
        if hr_img is None or lr_img is None:
            raise ValueError(f'无法读取: {self.hr_paths[idx]} / {self.lr_paths[idx]}')
        hr_img = cv2.cvtColor(hr_img, cv2.COLOR_BGR2RGB)
        lr_img = cv2.cvtColor(lr_img, cv2.COLOR_BGR2RGB)

        h_hr, w_hr, _ = hr_img.shape
        h_lr, w_lr, _ = lr_img.shape
        lps = self.patch_size // self.scale

        if h_hr < self.patch_size or w_hr < self.patch_size:
            hr_img = cv2.resize(hr_img, (max(w_hr, self.patch_size), max(h_hr, self.patch_size)))
            lr_img = cv2.resize(lr_img, (max(w_lr, lps), max(h_lr, lps)))
            h_hr, w_hr, _ = hr_img.shape
            h_lr, w_lr, _ = lr_img.shape

        x_lr = random.randint(0, w_lr - lps)
        y_lr = random.randint(0, h_lr - lps)
        x_hr, y_hr = x_lr * self.scale, y_lr * self.scale

        hr_patch = hr_img[y_hr:y_hr+self.patch_size, x_hr:x_hr+self.patch_size, :]
        lr_patch = lr_img[y_lr:y_lr+lps, x_lr:x_lr+lps, :]

        if hr_patch.size == 0 or lr_patch.size == 0:
            raise ValueError(f'裁剪失败: hr={hr_img.shape}, lr={lr_img.shape}, ps={self.patch_size}, lps={lps}')

        if random.random() < 0.5:
            hr_patch = cv2.flip(hr_patch, 1); lr_patch = cv2.flip(lr_patch, 1)
        if random.random() < 0.5:
            hr_patch = cv2.flip(hr_patch, 0); lr_patch = cv2.flip(lr_patch, 0)
        if random.random() < 0.5:
            hr_patch = hr_patch.transpose(1,0,2); lr_patch = lr_patch.transpose(1,0,2)

        hr_t = torch.from_numpy(hr_patch.transpose(2,0,1)).float() / 255.0
        lr_t = torch.from_numpy(lr_patch.transpose(2,0,1)).float() / 255.0
        return {'lr': lr_t, 'hr': hr_t}

train_dataset = AnimeSRDataset(LR_DIR, HR_DIR, patch_size=PATCH_SIZE, scale=SCALE)
train_loader  = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=2, pin_memory=True)
print(f'DataLoader 就绪: {len(train_dataset)} samples, {len(train_loader)} batches/epoch')

In [ ]:
# ============================================================
# 9. 模型 + 加载预训练权重
# ============================================================

generator     = RRDBNet(3, 3, scale=SCALE, num_block=NUM_BLOCK).to(device)
discriminator = UNetDiscriminatorSN(3).to(device)

pattern = os.path.join(PRETRAINED_DIR, '*4x*.pth')
pretrained_paths = sorted(glob.glob(pattern))
if not pretrained_paths:
    pretrained_paths = sorted(glob.glob(os.path.join(PRETRAINED_DIR, '*.pth')))

if pretrained_paths:
    pp = pretrained_paths[0]
    print(f'加载预训练: {pp}')
    ckpt = torch.load(pp, map_location=device)
    if 'params_ema' in ckpt:       state = ckpt['params_ema']
    elif 'model_state_dict' in ckpt: state = ckpt['model_state_dict']
    else:                            state = ckpt
    model_state = generator.state_dict()
    compatible = {k: v for k, v in state.items() if k in model_state and model_state[k].shape == v.shape}
    generator.load_state_dict(compatible, strict=False)
    print(f'匹配 {len(compatible)} / {len(state)} 参数 (跳过 {len(state)-len(compatible)})')
else:
    print('⚠ 未找到预训练权重，将从头训练。')

In [ ]:
# ============================================================
# 10. 损失函数 + 优化器 + 调度器
# ============================================================

cri_pix = nn.L1Loss().to(device)

cri_vgg = PerceptualLoss(
    layer_weights={'conv1_2': 0.1, 'conv2_2': 0.1, 'conv3_4': 1, 'conv4_4': 1, 'conv5_4': 1},
    vgg_type='vgg19',
    perceptual_weight=WEIGHT_VGG
).to(device)

cri_danbooru = Anime_PerceptualLoss(
    layer_weights={'0': 0.1, '4_2_conv3': 20, '5_3_conv3': 25, '6_5_conv3': 1, '7_2_conv3': 1},
    perceptual_weight=WEIGHT_DANBOORU
).to(device)

cri_gan = GANLoss(gan_type='vanilla', loss_weight=WEIGHT_GAN).to(device)

optimizer_g = optim.Adam(generator.parameters(),     lr=LEARNING_RATE, betas=(0.9, 0.999))
optimizer_d = optim.Adam(discriminator.parameters(),  lr=LEARNING_RATE, betas=(0.9, 0.999))

scheduler_g = optim.lr_scheduler.StepLR(optimizer_g, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)
scheduler_d = optim.lr_scheduler.StepLR(optimizer_d, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

scaler = torch.amp.GradScaler('cuda')

print('损失函数、优化器、调度器初始化完成。')

In [ ]:
# ============================================================
# 11. 训练主循环
# ============================================================

os.makedirs('saved_models', exist_ok=True)
assert os.path.isdir(LR_DIR), f'LR 路径不存在: {LR_DIR}'
assert os.path.isdir(HR_DIR), f'HR 路径不存在: {HR_DIR}'

print('开始训练...')
best_loss = float('inf')
history_loss_g, history_loss_d = [], []

for epoch in range(NUM_EPOCHS):
    generator.train()
    discriminator.train()
    epoch_loss_g, epoch_loss_d = 0.0, 0.0

    if epoch > 0:
        scheduler_g.step()
        scheduler_d.step()
        for opt in [optimizer_g, optimizer_d]:
            for pg in opt.param_groups:
                pg['lr'] = max(pg['lr'], LR_MIN)

    is_warmup = epoch < WARMUP_EPOCHS
    phase = 'Warmup (L1 only)' if is_warmup else 'GAN (L1+Percep+GAN)'
    print(f'--- Epoch {epoch+1}/{NUM_EPOCHS}: {phase} ---')

    pbar = tqdm(train_loader, desc=f'Epoch [{epoch+1}/{NUM_EPOCHS}]')
    for batch in pbar:
        imgs_lr = batch['lr'].to(device)
        imgs_hr = batch['hr'].to(device)

        # ---- Generator ----
        optimizer_g.zero_grad()
        for p in discriminator.parameters(): p.requires_grad = False

        with torch.amp.autocast('cuda'):
            gen_hr = generator(imgs_lr)
            l_pix = cri_pix(gen_hr, imgs_hr) * WEIGHT_PIXEL
            if is_warmup:
                loss_g = l_pix
                l_percep = torch.tensor(0.0, device=device)
                l_gan_g  = torch.tensor(0.0, device=device)
            else:
                l_percep = cri_vgg(gen_hr, imgs_hr) + cri_danbooru(gen_hr, imgs_hr)
                l_gan_g  = cri_gan(discriminator(gen_hr), True, is_disc=False)
                loss_g   = l_pix + l_percep + l_gan_g

        scaler.scale(loss_g).backward()
        scaler.step(optimizer_g)
        scaler.update()
        epoch_loss_g += loss_g.item()

        # ---- Discriminator ----
        if not is_warmup:
            for p in discriminator.parameters(): p.requires_grad = True
            optimizer_d.zero_grad()
            with torch.amp.autocast('cuda'):
                l_d_real = cri_gan(discriminator(imgs_hr),        True,  is_disc=True)
                l_d_fake = cri_gan(discriminator(gen_hr.detach()), False, is_disc=True)
                loss_d   = l_d_real + l_d_fake
            scaler.scale(loss_d).backward()
            scaler.step(optimizer_d)
            scaler.update()
            epoch_loss_d += loss_d.item()
        else:
            loss_d = torch.tensor(0.0, device=device)

        pbar.set_postfix(G=f'{loss_g.item():.4f}', D=f'{loss_d.item():.4f}',
                         L1=f'{l_pix.item():.4f}', P=f'{l_percep.item():.4f}', GAN=f'{l_gan_g.item():.4f}')

    # ---- Epoch 结束 ----
    avg_g = epoch_loss_g / len(train_loader)
    avg_d = epoch_loss_d / len(train_loader) if not is_warmup else 0.0
    history_loss_g.append(avg_g)
    history_loss_d.append(avg_d)
    cur_lr = optimizer_g.param_groups[0]['lr']
    print(f'Epoch {epoch+1} 完成 | G: {avg_g:.4f} | D: {avg_d:.4f} | LR: {cur_lr:.6f}')

    if avg_g < best_loss and not is_warmup:
        best_loss = avg_g
        bp = f'saved_models/{CHECKPOINT_NAME}_best.pth'
        torch.save({'epoch': epoch+1, 'model_state_dict': generator.state_dict(),
                     'optimizer_state_dict': optimizer_g.state_dict(), 'loss': avg_g}, bp)
        print(f'  → Best model saved: {bp}')

    if (epoch+1) % SAVE_FREQ == 0 or (epoch+1) == NUM_EPOCHS:
        sp = f'saved_models/{CHECKPOINT_NAME}_epoch_{epoch+1}.pth'
        torch.save({'epoch': epoch+1, 'model_state_dict': generator.state_dict(),
                     'optimizer_state_dict': optimizer_g.state_dict(), 'loss': avg_g}, sp)
        print(f'  → Checkpoint saved: {sp}')

print('\n训练完成！')

In [ ]:
# ============================================================
# 12. 绘制损失曲线
# ============================================================
if history_loss_g:
    epochs_range = range(1, len(history_loss_g)+1)
    plt.figure(figsize=(10, 5))
    plt.plot(epochs_range, history_loss_g, label='Generator Loss')
    plt.plot(epochs_range, history_loss_d, label='Discriminator Loss')
    plt.xlabel('Epoch'); plt.ylabel('Loss')
    plt.title('Training Loss Curve (4x / 23-block / simple)')
    plt.legend(); plt.grid(True)
    plt.savefig('saved_models/loss_curve_4x_23block_simple.png')
    plt.show()
else:
    print('无训练记录。')